# Семинар (практика). GPT / decoder-only Transformer: генерация, семплинг и простой RAG (GPU-friendly)

**Сквозная цель:** на практике понять, как устроена *decoder-only* модель (GPT-класс), как она **генерирует** текст, как влияют параметры **temperature / top-k / top-p / beam search / penalties** и как собрать **минимальный RAG-прототип** (retrieval → prompt → generation).

## Что получится в конце
- Вы умеете запускать GPT-подобную модель на GPU и управлять генерацией
- Вы понимаете, почему `do_sample=True` важно для `temperature/top_p/top_k`
- У вас есть рабочий **RAG прототип**: индекс → поиск → ответ с контекстом + история диалога

## План
1) Setup + GPU  
2) Мини-теория: causal mask + KV-cache  
3) Токенизация (BPE) на примере GPT-токенизатора  
4) Генерация и decoding-параметры (temperature/top-k/top-p и др.)  
5) (Опционально) Fine-tune маленького GPT (Causal LM) + perplexity  
6) RAG: SentenceTransformers + FAISS + GPT-Instruct  
7) Практическое задание

> Ссылки на открытые примеры семинаров/ноутбуков — в конце.

## 1) Зависимости (Colab-friendly)

⚠️ Если вы запускаете локально и всё уже установлено — эту клетку можно пропустить/закомментировать.

In [1]:
!pip -q install "transformers>=4.40" "datasets>=2.18" "accelerate>=0.29" "evaluate>=0.4" \
               "sentence-transformers>=2.6" "faiss-cpu" "scikit-learn" "numpy" "pandas" "matplotlib" "tqdm"


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, math, time, random, re
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
# === Проверка GPU ===
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
else:
    print("CUDA not available. Будет медленнее, но ноутбук всё равно работает.")

CUDA: 12.8
GPU: NVIDIA RTX A6000
VRAM (GB): 47.4


## 2) Мини-теория: почему GPT = decoder-only Transformer

**Decoder-only** трансформер обучается на задаче *next-token prediction*:  
по токенам `x_1..x_t` предсказать `x_{t+1}`

Ключевые детали:
- **Causal mask**: на позиции `t` модель видит только `<= t` (не смотрит в будущее)
- **KV-cache**: при генерации храним Key/Value прошлых токенов → ускоряем авто-регрессию

## 3) Токенизация GPT (BPE) на пальцах

GPT-семейство использует подсловную токенизацию (BPE-подобную).  
Модель "видит" не слова, а **токены**.

Проверим на примере `gpt2` токенизатора.

In [4]:
from transformers import AutoTokenizer

tok_gpt = AutoTokenizer.from_pretrained("gpt2")

examples = [
    "Profit rises today.",
    "unbelievable",
    "Bank of the river vs bank account",
    "Русский текст: прибыль выросла на 10%."
]

for t in examples:
    ids = tok_gpt.encode(t)
    toks = tok_gpt.convert_ids_to_tokens(ids)
    print("\nTEXT:", t)
    print("n_tokens:", len(ids))
    print("tokens:", toks)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


TEXT: Profit rises today.
n_tokens: 5
tokens: ['Pro', 'fit', 'Ġrises', 'Ġtoday', '.']

TEXT: unbelievable
n_tokens: 4
tokens: ['un', 'bel', 'iev', 'able']

TEXT: Bank of the river vs bank account
n_tokens: 7
tokens: ['Bank', 'Ġof', 'Ġthe', 'Ġriver', 'Ġvs', 'Ġbank', 'Ġaccount']

TEXT: Русский текст: прибыль выросла на 10%.
n_tokens: 38
tokens: ['Ð', 'ł', 'Ñĥ', 'Ñģ', 'Ñģ', 'Ðº', 'Ð¸', 'Ð', '¹', 'Ġ', 'ÑĤ', 'Ðµ', 'Ðº', 'Ñģ', 'ÑĤ', ':', 'ĠÐ', '¿', 'ÑĢ', 'Ð¸', 'Ð', '±', 'Ñĭ', 'Ð»', 'ÑĮ', 'ĠÐ', '²', 'Ñĭ', 'ÑĢ', 'Ð¾', 'Ñģ', 'Ð»', 'Ð°', 'ĠÐ', '½', 'Ð°', 'Ġ10', '%.']


## 4) Генерация текста: параметры, которые реально меняют поведение

Берём **маленькую instruct-модель (decoder-only)**.

Параметры генерации:
- `max_new_tokens` — сколько новых токенов генерировать
- `do_sample=False` → greedy/beam; `do_sample=True` → sampling
- `temperature`, `top_k`, `top_p` — управление разнообразием (работают только при `do_sample=True`)
- `num_beams` — beam search
- `repetition_penalty`, `no_repeat_ngram_size` — борьба с повторами

Документация:
- generation parameters: https://huggingface.co/docs/transformers/en/main_classes/text_generation  
- generation strategies: https://huggingface.co/docs/transformers/en/generation_strategies  
- про `do_sample=True`: https://github.com/huggingface/transformers/issues/22405

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# Можно заменить на:
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

def model_device(m):
    try:
        return next(m.parameters()).device
    except StopIteration:
        return torch.device("cpu")

print("Loaded:", MODEL_NAME)
print("Device:", model_device(model))
print("Has chat template:", tok.chat_template is not None)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda:0
Has chat template: True


In [6]:
def build_chat_prompt(messages, tokenizer):
    # messages: list of dicts {"role": "...", "content": "..."}
    # Возвращаем строку prompt для chat-моделей (если есть chat_template),
    # иначе используем простой текстовый формат.
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    out = []
    for m in messages:
        out.append(f"{m['role'].upper()}: {m['content']}")
    out.append("ASSISTANT:")
    return "\n".join(out)

@torch.no_grad()
def generate_chat(
    messages,
    max_new_tokens=160,
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    top_k=50,
    repetition_penalty=1.05,
    num_beams=1,
    no_repeat_ngram_size=0,
):
    # ✅ корректно отделяем ответ от prompt: декодируем только новые токены
    prompt = build_chat_prompt(messages, tok)
    inputs = tok(prompt, return_tensors="pt").to(model_device(model))

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=repetition_penalty,
        num_beams=num_beams,
        no_repeat_ngram_size=no_repeat_ngram_size,
        use_cache=True,
        pad_token_id=tok.eos_token_id,
    )

    out = model.generate(**inputs, **gen_kwargs)
    new_tokens = out[0, inputs["input_ids"].shape[-1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()

def ask(prompt, **kwargs):
    messages = [
        {"role":"system", "content":"Ты полезный ассистент. Отвечай кратко и по делу."},
        {"role":"user", "content": prompt},
    ]
    return generate_chat(messages, **kwargs)

print(ask("Объясни разницу между encoder и decoder в трансформере в 3 пунктах."))

1. **Цель и работа encoder**: 
   - Константин Каменев и Сергей Коробов обсудили принцип работы encoder, который преобразует исходное значение в качестве в двоичную систему и отправляет его к decoder.
   
2. **Цель и работа decoder**: 
   - Константин Каменев и Сергей Коробов также обсудили работу decoder, которое преобразует результаты encoder в формат данных, которые могут быть использованы для обработки и дальнейшей работы программы.

3. **Процесс работы**: 
   - Каждый из этих процессов основывается на взаимодействии двух моделей: encoder и decoder. Encoder преобразует входящее значение


### 4.1 Greedy vs Sampling

In [7]:
prompt = "Придумай короткое объяснение, что такое top_p и top_k, на примере генерации текста."

print("=== Greedy (do_sample=False) ===")
print(ask(prompt, do_sample=False, num_beams=1, max_new_tokens=140))

print("\n=== Sampling (t=0.8, top_p=0.95, top_k=50) ===")
print(ask(prompt, do_sample=True, temperature=0.8, top_p=0.95, top_k=50, max_new_tokens=140))

print("\n=== Более 'креативно' (t=1.2) ===")
print(ask(prompt, do_sample=True, temperature=1.2, top_p=0.95, top_k=50, max_new_tokens=140))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Greedy (do_sample=False) ===
Top_P и Top_K - это два метода, используемых в машинном обучении для выбора наиболее вероятных ответов или предложений. 

1. **Top_P**: Этот метод выбирает k самых вероятных ответов или предложений из n возможных. Он помогает улучшить качество ответов, особенно при наличии многочисленных вариантов ответов.

2. **Top_K**: Этот метод выбирает k самых вероятных ответов или предложений из n возможных. Он также помогает улучшить качество ответов, особенно при наличии многочисленных вариантов ответов.

Эти методы используются в

=== Sampling (t=0.8, top_p=0.95, top_k=50) ===
Top_P (Highest probability) — это метод машинного обучения, который выбирает наиболее вероятные значения отдельных аргументов для модели, чтобы выявить наиболее важные и релевантные ответы или предсказания. Такие агентства могут быть использованы для оптимизации логистических маршрутов, предборку популярности контента или распознавания предпочтений пользователей.

Top_K (Kth most likely) 

### 4.2 Температура / top_p / top_k: быстрые тесты

In [8]:
settings = [
    dict(name="t=0.3, top_p=1.0 (почти детерминир.)", temperature=0.3, top_p=1.0, top_k=0),
    dict(name="t=0.8, top_p=0.95", temperature=0.8, top_p=0.95, top_k=50),
    dict(name="t=1.1, top_p=0.9 (больше разнообразия)", temperature=1.1, top_p=0.9, top_k=50),
]

prompt = "Сгенерируй 3 предложения о том, почему causal mask нужен в GPT."

for s in settings:
    print("\n---", s["name"], "---")
    print(ask(prompt, do_sample=True, max_new_tokens=120, **{k:v for k,v in s.items() if k!='name'}))


--- t=0.3, top_p=1.0 (почти детерминир.) ---
1. Для улучшения точности и контекстной информации в тексте.
2. Для предотвращения синтаксических ошибок и недоразумений.
3. Для создания более читаемого и понятного текста.

--- t=0.8, top_p=0.95 ---
1. Causal masks help to prevent overfitting by masking important features from the input data, which makes it easier for the model to generalize and make accurate predictions.

2. In machine learning, causal masks are essential to avoid bias in predictions by preventing specific effects that might occur based on the relationship between inputs and outputs.

3. For applications requiring sophisticated tasks like natural language processing or computer vision, causal masks provide a critical component to ensure the model’s predictive power is robust against potential biases, ensuring accurate outcomes despite the input's inherent complexity.

--- t=1.1, top_p=0.9 (больше разнообразия) ---
В GPT (Generalized Answering Transformer) есть сеть, кото

### 4.3 Борьба с повторами

In [9]:
prompt = "Напиши 6 пунктов, что может пойти не так при генерации текста (LLM)."

print("=== Без ограничений ===")
print(ask(prompt, do_sample=True, temperature=0.9, top_p=0.95, max_new_tokens=180, repetition_penalty=1.0, no_repeat_ngram_size=0))

print("\n=== repetition_penalty=1.15 + no_repeat_ngram_size=3 ===")
print(ask(prompt, do_sample=True, temperature=0.9, top_p=0.95, max_new_tokens=180, repetition_penalty=1.15, no_repeat_ngram_size=3))

=== Без ограничений ===
1. Слишком много предложений или устойчивые привычки
2. Всякий раз переопределять предыдущие впечатления
3. Отсутствие определения определенных аспектов
4. Императивность и безусловные последовательности
5. Недостаток контекста для объяснения
6. Регулярное повторение предыдущих текста

=== repetition_penalty=1.15 + no_repeat_ngram_size=3 ===
1. Неправильные словообразование: Влияние на создание словарных или нейросетевых ресурсов.
2. Невозможно достичь конечного значения: Масштабирование запросов модели.
3. Блокировка уникальности: Стратегии разработки для ограничения повторений.
4. Помощь в тексте-обучении: Обработка различных типов данных.
5. Негативное влияние: Изменение результата обучающей функции.
6. Удаленные возможности: Создание новых языковых систем.


### 4.4 Скорость: KV-cache (очень грубо)

In [10]:
@torch.no_grad()
def bench(prompt, n_runs=3, max_new_tokens=128):
    messages = [{"role":"user","content": prompt}]
    p = build_chat_prompt(messages, tok)
    inputs = tok(p, return_tensors="pt").to(model_device(model))

    _ = model.generate(**inputs, max_new_tokens=16, do_sample=False, use_cache=True, pad_token_id=tok.eos_token_id)

    times = []
    for _ in range(n_runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.time()
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True, pad_token_id=tok.eos_token_id)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.time()
        n_new = out.shape[-1] - inputs["input_ids"].shape[-1]
        times.append((t1-t0, n_new))
    return times

prompt = "Объясни, что такое KV-cache в декодере и почему он ускоряет генерацию."
times = bench(prompt, n_runs=3, max_new_tokens=160)

for dt, n_new in times:
    print(f"time={dt:.3f}s, new_tokens={n_new}, tok/s={n_new/dt:.1f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


time=0.858s, new_tokens=160, tok/s=186.4
time=0.846s, new_tokens=160, tok/s=189.1
time=0.843s, new_tokens=160, tok/s=189.8


## 5) Fine-tuning маленького GPT (Causal LM) на WikiText-2 (опционально)

Цель: увидеть causal LM и посчитать perplexity = exp(loss).  
HF notebook: https://github.com/huggingface/notebooks/blob/main/examples/language_modeling.ipynb

In [11]:
from datasets import load_dataset

ds = load_dataset("wikitext", "wikitext-2-raw-v1")
ds

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [12]:
print(ds["train"][0]["text"][:400])

In [13]:
from transformers import AutoTokenizer as HFAutoTokenizer
from transformers import AutoModelForCausalLM as HFAutoModelForCausalLM
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

LM_MODEL = "distilgpt2"
lm_tok = HFAutoTokenizer.from_pretrained(LM_MODEL)
lm_tok.pad_token = lm_tok.eos_token

block_size = 128

def tokenize_function(examples):
    return lm_tok(examples["text"])

tokenized = ds.map(tokenize_function, batched=True, remove_columns=["text"])

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size
    result = {k: [t[i:i+block_size] for i in range(0, total_length, block_size)] for k,t in concatenated.items()}
    result["labels"] = result["input_ids"].copy()
    return result

lm_train = tokenized["train"].select(range(min(4000, len(tokenized["train"]))))
lm_val   = tokenized["validation"].select(range(min(1000, len(tokenized["validation"]))))

lm_train = lm_train.map(group_texts, batched=True)
lm_val   = lm_val.map(group_texts, batched=True)

(len(lm_train), len(lm_val), lm_train[0]["input_ids"][:10])

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

(2046, 426, [796, 569, 18354, 7496, 17740, 6711, 796, 220, 198, 2311])

In [16]:
lm_model = HFAutoModelForCausalLM.from_pretrained(
    LM_MODEL,
    torch_dtype=torch.float32,   # <-- важно
).to(device)

data_collator = DataCollatorForLanguageModeling(tokenizer=lm_tok, mlm=False)

args = TrainingArguments(
    output_dir="./gpt_lm_finetune",
    per_device_train_batch_size=8 if torch.cuda.is_available() else 2,
    per_device_eval_batch_size=8 if torch.cuda.is_available() else 2,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to=[],
)

trainer = Trainer(
    model=lm_model,
    args=args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=data_collator,
)

_ = trainer.train()
eval_out = trainer.evaluate()

loss = eval_out["eval_loss"]
ppl = math.exp(loss)
print("eval_loss:", loss, "perplexity:", ppl)

Epoch,Training Loss,Validation Loss
1,3.892500,3.669630


eval_loss: 3.6696300506591797 perplexity: 39.23738732933161


In [17]:
@torch.no_grad()
def lm_generate(prompt, max_new_tokens=80, temperature=0.9, top_p=0.95):
    inp = lm_tok(prompt, return_tensors="pt").to(device)
    out = lm_model.generate(
        **inp,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=lm_tok.eos_token_id
    )
    return lm_tok.decode(out[0], skip_special_tokens=True)

print(lm_generate("The company said", max_new_tokens=60))

The company said the news was due to be released in October , but a further delay may result in delays in distribution of the film . At the same time , the film was also shown in the UK on DVD . In October 2007 , the film was released for the U.S., Canada and the US on Blu


## 6) Мини-RAG прототип: retrieve → prompt → generate

RAG = retrieval-augmented generation:
1) индексируем документы эмбеддингами  
2) для вопроса берём top-k чанков  
3) строим prompt с `CONTEXT`  
4) генерируем ответ

RAG-референс:
- LLM Zoomcamp: https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/rag-intro.ipynb  
- HF Cookbook Advanced RAG: https://huggingface.co/learn/cookbook/advanced_rag

In [18]:
from sentence_transformers import SentenceTransformer
import faiss

EMB_MODEL = "intfloat/multilingual-e5-small"
emb = SentenceTransformer(EMB_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
print("Embedding model:", EMB_MODEL)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Embedding model: intfloat/multilingual-e5-small


In [19]:
from datasets import load_dataset

squad = load_dataset("squad")
N_CTX = 800
N_Q = 40

ctxs = squad["train"].select(range(N_CTX))
qs = squad["validation"].select(range(N_Q))

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [20]:
def split_into_chunks(text, max_chars=450):
    sents = re.split(r"(?<=[\.\!\?])\s+", text.strip())
    chunks, cur = [], ""
    for s in sents:
        if not s:
            continue
        if len(cur) + len(s) + 1 <= max_chars:
            cur = (cur + " " + s).strip()
        else:
            if cur:
                chunks.append(cur)
            cur = s.strip()
    if cur:
        chunks.append(cur)
    return chunks

docs = []
for ex in ctxs:
    for ch in split_into_chunks(ex["context"], max_chars=450):
        docs.append(ch)

print("n_chunks:", len(docs))
print("sample:\n", docs[0][:300])

n_chunks: 2200
sample:
 Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is 


In [21]:
def embed_passages(passages, batch_size=64):
    texts = ["passage: " + p for p in passages]
    vecs = emb.encode(texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
    return vecs.astype("float32")

def embed_query(q):
    vec = emb.encode(["query: " + q], normalize_embeddings=True)
    return vec.astype("float32")

doc_vecs = embed_passages(docs, batch_size=64)
index = faiss.IndexFlatIP(doc_vecs.shape[1])
index.add(doc_vecs)
print("FAISS size:", index.ntotal)

Batches:   0%|          | 0/35 [00:00<?, ?it/s]

FAISS size: 2200


In [22]:
def retrieve(query, k=4):
    qv = embed_query(query)
    scores, idx = index.search(qv, k)
    return [(docs[i], float(scores[0][j])) for j,i in enumerate(idx[0])]

q_demo = qs[0]["question"]
top = retrieve(q_demo, k=4)
print("Q:", q_demo)
for i,(d,s) in enumerate(top, 1):
    print(f"\n[{i}] score={s:.3f}\n{d[:350]}")

Q: Which NFL team represented the AFC at Super Bowl 50?

[1] score=0.789
Men's sports include baseball, basketball, crew, cross country, fencing, football, golf, ice hockey, lacrosse, soccer, swimming & diving, tennis and track & field; while women's sports include basketball, cross country, fencing, golf, lacrosse, rowing, soccer, softball, swimming & diving, tennis, track & field and volleyball. The football team comp

[2] score=0.789
Men's sports include baseball, basketball, crew, cross country, fencing, football, golf, ice hockey, lacrosse, soccer, swimming & diving, tennis and track & field; while women's sports include basketball, cross country, fencing, golf, lacrosse, rowing, soccer, softball, swimming & diving, tennis, track & field and volleyball. The football team comp

[3] score=0.789
Men's sports include baseball, basketball, crew, cross country, fencing, football, golf, ice hockey, lacrosse, soccer, swimming & diving, tennis and track & field; while women's sports includ

In [23]:
def rag_answer(question, k=4, max_new_tokens=200, temperature=0.3, top_p=0.9, min_score=None):
    retrieved = retrieve(question, k=k)

    if min_score is not None and (len(retrieved)==0 or max(s for _,s in retrieved) < min_score):
        return "Не знаю по контексту (низкая уверенность retrieval).", retrieved

    context = "\n\n".join([f"[{i+1}] {txt}" for i,(txt,_) in enumerate(retrieved)])

    messages = [
        {"role":"system","content":"Ты отвечаешь на вопрос ТОЛЬКО используя предоставленный контекст. Если ответа нет — скажи 'Не знаю по контексту'."},
        {"role":"user","content": f"CONTEXT:\n{context}\n\nQUESTION: {question}\n\nОтветь кратко, 1-3 предложения. При необходимости укажи источники [1],[2]."}
    ]
    ans = generate_chat(messages, do_sample=True, temperature=temperature, top_p=top_p, top_k=50, max_new_tokens=max_new_tokens)
    return ans, retrieved

q = qs[0]["question"]
gold = qs[0]["answers"]["text"][0] if len(qs[0]["answers"]["text"]) else ""
ans, _ = rag_answer(q, k=4)

print("QUESTION:", q)
print("GOLD:", gold)
print("\nRAG ANSWER:\n", ans)

QUESTION: Which NFL team represented the AFC at Super Bowl 50?
GOLD: Denver Broncos

RAG ANSWER:
 The football team representing the AFC at Super Bowl 50 was the New England Patriots. This can be inferred from the information provided in the context.


In [24]:
q = qs[1]["question"]
gold = qs[1]["answers"]["text"][0] if len(qs[1]["answers"]["text"]) else ""

no_rag = ask(f"Вопрос: {q}\nОтветь кратко. Если не знаешь — скажи 'не знаю'.", temperature=0.3, top_p=0.9, max_new_tokens=120)
with_rag, _ = rag_answer(q, k=4, temperature=0.3, top_p=0.9, max_new_tokens=120)

print("QUESTION:", q)
print("GOLD:", gold)
print("\nNO RAG:\n", no_rag)
print("\nWITH RAG:\n", with_rag)

QUESTION: Which NFL team represented the NFC at Super Bowl 50?
GOLD: Carolina Panthers

NO RAG:
 The New York Giants represented the NFC at Super Bowl 50.

WITH RAG:
 Super Bowl 50 was played between the New England Patriots and the Pittsburgh Steelers. The New England Patriots represented the NFC.


In [25]:
chat_history = [
    {"role":"user","content":"Мы обсуждаем финансовые новости и тональность текстов."},
    {"role":"assistant","content":"Окей, я помогу с вопросами по тональности и финансовому контексту."},
    {"role":"user","content":"Важно отвечать только по контексту, без догадок."},
]

def rag_answer_with_history(question, history, k=4, history_turns=3):
    retrieved = retrieve(question, k=k)
    context = "\n\n".join([f"[{i+1}] {txt}" for i,(txt,_) in enumerate(retrieved)])
    hist = history[-history_turns:]

    messages = [
        {"role":"system","content":"Ты отвечаешь на вопрос ТОЛЬКО используя предоставленный контекст. Если ответа нет — скажи 'Не знаю по контексту'."},
        *hist,
        {"role":"user","content": f"CONTEXT:\n{context}\n\nQUESTION: {question}\n\nОтветь кратко."}
    ]
    ans = generate_chat(messages, do_sample=True, temperature=0.3, top_p=0.9, top_k=50, max_new_tokens=160)
    return ans, retrieved

q = qs[2]["question"]
gold = qs[2]["answers"]["text"][0] if len(qs[2]["answers"]["text"]) else ""
ans_hist, _ = rag_answer_with_history(q, chat_history, k=4)

print("QUESTION:", q)
print("GOLD:", gold)
print("\nRAG+HISTORY:\n", ans_hist)

QUESTION: Where did Super Bowl 50 take place?
GOLD: Santa Clara, California

RAG+HISTORY:
 Houston, Texas


In [26]:
# Мини-оценка SQuAD (опционально). Может не сработать без скачивания evaluate/метрик.
try:
    import evaluate
    squad_metric = evaluate.load("squad")

    def clean_pred(s):
        s = s.strip()
        s = re.sub(r"^(Ответ|Answer)\s*:\s*", "", s, flags=re.I)
        return s

    preds, refs = [], []
    for i in tqdm(range(min(N_Q, 20))):
        q = qs[i]["question"]
        golds = qs[i]["answers"]["text"]
        if not golds:
            continue
        pred, _ = rag_answer(q, k=4, temperature=0.2, top_p=0.9, max_new_tokens=120)
        preds.append({"id": str(i), "prediction_text": clean_pred(pred)})
        refs.append({"id": str(i), "answers": {"text": golds, "answer_start": qs[i]["answers"]["answer_start"]}})

    print(squad_metric.compute(predictions=preds, references=refs))
except Exception as e:
    print("Пропуск метрик. Ошибка:", repr(e))

  0%|          | 0/20 [00:00<?, ?it/s]

{'exact_match': 0.0, 'f1': 9.21161191749427}


## 7) Практическое задание (мини-проект)

1) Sampling-лаборатория  
   - 3 промпта (фактологический / творческий / "инструкция")  
   - Сравните greedy vs sampling, temperature=0.2/0.8/1.2, top_p=0.9 vs 0.98, top_k=20 vs 100  
   - Запишите наблюдения: повторения, галлюцинации, фактичность.

2) Fine-tune (по желанию)  
   - Подставьте свой корпус, попробуйте `block_size=256`  
   - Сравните perplexity и 2-3 примера генерации.

3) RAG  
   - Подставьте свой корпус "сообщений" (список строк)  
   - Варьируйте `k=2/4/8`, добавьте threshold по score → "не знаю".

---

## Открытые референсы (как делают семинары/ноутбуки)

- HF generation params: https://huggingface.co/docs/transformers/en/main_classes/text_generation  
- HF generation strategies: https://huggingface.co/docs/transformers/en/generation_strategies  
- HF notebook (LM fine-tune): https://github.com/huggingface/notebooks/blob/main/examples/language_modeling.ipynb  
- LLM Zoomcamp RAG: https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/rag-intro.ipynb  
- HF Cookbook Advanced RAG: https://huggingface.co/learn/cookbook/advanced_rag  
- Karpathy build-nanogpt: https://github.com/karpathy/build-nanogpt  
- nanoGPT: https://github.com/karpathy/nanoGPT  
- Transformer-from-scratch (tiny GPT): https://github.com/sajalregmi/transformer-from-scratch